In [65]:
%pip install tensorflow nltk gensim

Note: you may need to restart the kernel to use updated packages.


You should consider upgrading via the 'c:\Users\Velicia\AppData\Local\Programs\Python\Python310\python.exe -m pip install --upgrade pip' command.


In [66]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
import tensorflow as tf
from tensorflow.keras import Sequential, callbacks, layers, optimizers, Model
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.layers import LSTM, Dense, Dropout, Bidirectional
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
import joblib
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix
from sklearn.preprocessing import LabelEncoder, StandardScaler
import re
import nltk
import gensim
from gensim.models import Word2Vec
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
nltk.download('punkt')
nltk.download('stopwords')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Velicia\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Velicia\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [67]:
cv_df = pd.read_csv('cv_processed.csv')
job_df = pd.read_csv('JobMarket_Preprocessed.csv')

In [68]:
cv_df.head()

,Unnamed: 0,user_id,raw_cv_text,extracted_user_skills,last_role,education,major,extracted_skills_structured,projects,experience
0,0,1,one97 communication limited data scientist jan...,object detection market research natural langu...,data scientist jan 2019 to till date,probability estimate premium amount charged bt...,"computer science, engineering, data science","python, java, machine learning, deep learning,...",expand arsenal application building work diffe...,limited data scientist jan 2019 till date | co...
1,1,2,attila lászló joó phd external advisor phd civ...,virtual design independent research and develo...,phd civil engineer,attila lászló joó phd external advisor phd civ...,"engineering, civil engineering","analysis, model, engineering, design",bridge numerical modelling steel joint virtual...,phd civil engineer present workplace assistant...
2,2,3,vineet mathradas way 3023 house 1918 po box 60...,business reporting revenue kingdom company mai...,vineet\tmathradas\t\tway\tno\t3023\thouse\tno\...,2017 london school economics political science...,"business, finance, economics","analysis, model",comprehensive financial model comprising annua...,class india senior school examination grade 12...
3,3,4,ákos pohl chief executive manager msc civil en...,marital status design cleaning msc optimal sol...,civil engineer ceos civil engineering optimal...,email apohlce oseu qualification msc diploma c...,"engineering, civil engineering, economics","excel, engineering, design",isd dunaferr project 2008 hexagon team award 2...,chief executive manager msc civil engineering ...
4,4,5,resume name rama krushna dash post applied mec...,medical marital status notification pneumatic ...,post applied for ...,berhampur odisha 71 recently completed btech ...,"engineering, mechanical engineering","engineering, design",located centrally project site ii preventive m...,applied mechanical engineer contact 9078684961...


In [69]:
job_df.head()

,job_id,job_title,extracted_job_skills,experienced_level,extracted_job_skills_count,role_category,extracted_requirements,extracted_responsibilities
0,1,data analyst,sql,entry level,1,data analyst,s1 teknik industri informatika sains fresh gra...,menarik data dari database dan menyajikan seca...
1,2,data analyst/intelligence,power bi tableau excel,mid level,3,data analyst,bachelors degree in finance accounting economi...,build maintain and automate dashboards daily w...
2,3,data analyst,spss sql python power bi oracle excel,unspecified,6,other,data analyst 1 mengerti bahasa pemrograman bah...,data analyst berikut merupakan job desk data a...
3,4,data analyst associate,sql tableau python power bi oracle,senior level,5,data analyst,craft sql scripts examine source data structur...,get to know the team you will join a dynamic t...
4,5,data analyst officer for finance divison,excel,mid level,1,data analyst,bachelors degree any discipline with preferenc...,supporting accounting division in preparing th...


### Tokenize

In [70]:
cv_skills = cv_df['extracted_skills_structured'].tolist()
job_skills = job_df['extracted_job_skills'].tolist()
all_skills = cv_skills + job_skills
tokenizer = Tokenizer(num_words=10000, oov_token='<OOV>')
tokenizer.fit_on_texts(all_skills)

In [71]:
MAX_SEQUENCE_LENGTH = 256

def encode(text):
    sequences = tokenizer.texts_to_sequences([text])
    padded = pad_sequences(sequences, maxlen=MAX_SEQUENCE_LENGTH, padding='post', truncating='post')
    return padded[0]

cv_df['encoded_skills'] = cv_df['extracted_skills_structured'].apply(encode)
job_df['encoded_skills'] = job_df['extracted_job_skills'].apply(encode)

### Model

In [72]:
x_cv = np.array(cv_df['encoded_skills'].tolist())
x_job = np.array(job_df['encoded_skills'].tolist())
x = np.vstack([x_cv, x_job])
y = np.array([1] * len(x_cv) + [0] * len(x_job))

x_train, x_val, y_train, y_val = train_test_split(x, y, test_size=0.2, random_state=42)

In [73]:
embedding_dim = 300
vocab_size = 10000

inp = layers.Input(shape=(None,), dtype="int32")
x = layers.Embedding(vocab_size, embedding_dim, mask_zero=True)(inp)
x = layers.Bidirectional(layers.LSTM(128, return_sequences=True))(x)
x = layers.Bidirectional(layers.LSTM(64))(x)
x = layers.Dense(64, activation='relu')(x)
x = layers.Dropout(0.2)(x)
x = layers.Dense(1, activation='sigmoid')(x)
model = Model(inp, x, name="skillscout_nusantara")

In [74]:
callbacks_custom = [
    EarlyStopping(
        monitor='val_loss',
        patience=5,
        restore_best_weights=True,
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=3,
        min_lr=1e-7,
        verbose=1
    ),
]

In [75]:
model.compile(
    loss='binary_crossentropy', 
    optimizer=optimizers.Adam(learning_rate=0.01), 
    metrics=['accuracy']
)

In [76]:
history = model.fit(
    x_train, y_train,
    validation_data=(x_val, y_val),
    epochs=50,
    batch_size=32,
    callbacks=callbacks_custom,
    verbose=1
)

Epoch 1/50
41/41 ━━━━━━━━━━━━━━━━━━━━ 21s 411ms/step - accuracy: 0.9793 - loss: 0.0666 - val_accuracy: 1.0000 - val_loss: 0.0034 - learning_rate: 0.0100
Epoch 2/50
41/41 ━━━━━━━━━━━━━━━━━━━━ 16s 392ms/step - accuracy: 0.9992 - loss: 0.0067 - val_accuracy: 1.0000 - val_loss: 4.6397e-05 - learning_rate: 0.0100
Epoch 3/50
41/41 ━━━━━━━━━━━━━━━━━━━━ 16s 383ms/step - accuracy: 1.0000 - loss: 1.2971e-04 - val_accuracy: 1.0000 - val_loss: 5.9317e-06 - learning_rate: 0.0100
Epoch 4/50
41/41 ━━━━━━━━━━━━━━━━━━━━ 16s 393ms/step - accuracy: 1.0000 - loss: 4.4451e-05 - val_accuracy: 1.0000 - val_loss: 2.0715e-06 - learning_rate: 0.0100
Epoch 5/50
41/41 ━━━━━━━━━━━━━━━━━━━━ 0s 374ms/step - accuracy: 1.0000 - loss: 8.0640e-06
Epoch 5: ReduceLROnPlateau reducing learning rate to 0.004999999888241291.
41/41 ━━━━━━━━━━━━━━━━━━━━ 16s 399ms/step - accuracy: 1.0000 - loss: 1.5480e-05 - val_accuracy: 1.0000 - val_loss: 1.5081e-06 - learning_rate: 0.0100
Epoch 6/50
41/41 ━━━━━━━━━━━━━━━━━━━━ 16s 389ms/step 

In [89]:
from sklearn.metrics.pairwise import cosine_similarity

# Ambil sample CV dan Job skills
cv_skills = cv_df['encoded_skills'].iloc[0]  # CV pertama
job_skills = job_df['encoded_skills'].iloc[10]  # Job pertama

# Hitung similarity dengan cosine
similarity = cosine_similarity([cv_skills], [job_skills])[0][0]
similarity_percentage = similarity * 100

print(f"Similarity Score: {similarity_percentage:.2f}%")
job_skills = job_df.iloc[10]
job_skills

Similarity Score: 45.06%


job_id                                                                       11
job_title                                                   junior data analyst
extracted_job_skills                        sql tableau python r power bi excel
experienced_level                                                   unspecified
extracted_job_skills_count                                                    6
role_category                                                      data analyst
extracted_requirements        bachelors degree in data science statistics co...
extracted_responsibilities    as the data analyst at bnc you will play a cru...
encoded_skills                [8, 11, 12, 13, 9, 10, 3, 0, 0, 0, 0, 0, 0, 0,...
Name: 10, dtype: object

In [86]:
cv1 = cv_df.iloc[0]
cv1

Unnamed: 0                                                                     0
user_id                                                                        1
raw_cv_text                    one97 communication limited data scientist jan...
extracted_user_skills          object detection market research natural langu...
last_role                                   data scientist jan 2019 to till date
education                      probability estimate premium amount charged bt...
major                                computer science, engineering, data science
extracted_skills_structured    python, java, machine learning, deep learning,...
projects                       expand arsenal application building work diffe...
experience                     limited data scientist jan 2019 till date | co...
encoded_skills                 [12, 20, 49, 38, 50, 38, 51, 52, 16, 64, 16, 2...
Name: 0, dtype: object